# HiP-VesselAdapt (Kaggle version)

**Physics-Grounded Adaptation of Foundation Vessel Models for Ex-Vivo HiP-CT Imaging**

Rebuilt for Kaggle Notebooks (moved from Colab due to disk/GPU quota issues).
Kaggle advantages used here: attached datasets don't count against your storage quota and don't need re-downloading every session; `/kaggle/working/` output persists across sessions once you click **Save Version**.

## ⚠️ After ANY session restart, do this first:
1. Run the **SETUP CELL** below (rebuilds model + all functions/classes). Takes ~1-2 min.
2. Confirm datasets are attached in the right panel: `kidney_complete-organ` (your raw K292 upload) and `blood-vessel-segmentation` (Kaggle SenNet+HOA ground-truth data -- add via **+ Add Input** if not already attached).

## Pipeline stages in this notebook
1. Convert K292 raw `.jp2` slices -> `.nii.gz` volume (only needs to run once per session)
2. vesselFM zero-shot inference on K292
3. Collapse detection: `compute_collapse_score` (eccentricity + thickness + Hessian + vesselness)
4. `PhysicsAwareAdapter` + `CollapseAwareLoss` (PyTorch, frozen-backbone LoRA-style)
5. Real ground-truth data (Kaggle SenNet+HOA) via `TiffKidneyVesselDataset`
6. **(Not yet done)** Real few-shot training, baseline comparison, statistics, Murray's Law validation


## Step 0 -- check what's attached

In [ ]:
import os

print("Contents of /kaggle/input:")
for d in os.listdir("/kaggle/input"):
    print(" -", d)


## Step 1 -- install dependencies + clone vesselFM

In [ ]:
import os

if not os.path.exists("/kaggle/working/vesselFM"):
    os.system("git clone https://github.com/bwittmann/vesselFM /kaggle/working/vesselFM")

os.system("pip install hydra-core omegaconf -q")
os.system("pip install -e /kaggle/working/vesselFM -q")
os.system("pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121 -q")

print("Done. If torch was just reinstalled, go to Run -> Restart Session now, then continue from Cell 3.")

In [ ]:
import torch
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## Step 2 -- convert K292 raw `.jp2` slices to a `.nii.gz` volume

Adjust `jp2_dir` below to match the actual folder name under `/kaggle/input/` (check the Step 0 output above).

In [ ]:
import os

base = "/kaggle/input/datasets/rabeya100x"
for root, dirs, files in os.walk(base):
    depth = root.replace(base, "").count(os.sep)
    if depth > 2:  # don't go too deep, just enough to see structure
        dirs[:] = []
        continue
    print(root)
    for f in files[:5]:
        print("   ", f)

In [ ]:
from pathlib import Path
import numpy as np
import nibabel as nib
from PIL import Image
from tqdm import tqdm
import os

jp2_dir = Path("/kaggle/input/datasets/rabeya100x/163-52um-k292-kidney-complete-organ")
slice_files = sorted(jp2_dir.glob("*.jp2"))
print(f"Total slices found: {len(slice_files)}")

first_slice = np.array(Image.open(slice_files[0]))
print("Single slice shape:", first_slice.shape, "dtype:", first_slice.dtype)

In [ ]:
os.makedirs("/kaggle/working/real_input", exist_ok=True)
nii_path = "/kaggle/working/real_input/kidney_K292_163um.nii.gz"

if os.path.exists(nii_path):
    print("nii.gz already exists, skipping conversion.")
else:
    h, w = first_slice.shape
    d = len(slice_files)
    print(f"Volume shape will be: ({h}, {w}, {d})")

    volume = np.zeros((h, w, d), dtype=np.uint16)
    for i, f in enumerate(tqdm(slice_files, desc="Loading slices")):
        volume[:, :, i] = np.array(Image.open(f))

    nifti_img = nib.Nifti1Image(volume.astype(np.float32), affine=np.eye(4))
    nib.save(nifti_img, nii_path)
    print("Saved:", nii_path)

print("File size (MB):", os.path.getsize(nii_path) / 1e6)

## Step 3 -- run vesselFM zero-shot inference on K292

In [ ]:


os.makedirs("/kaggle/working/real_output", exist_ok=True)

if os.path.exists("/kaggle/working/real_output/kidney_K292_163um_pred.nii.gz"):
    print("Prediction already exists, skipping inference.")
else:
    exit_code = os.system(
        "cd /kaggle/working/vesselFM && python -m vesselfm.seg.inference "
        "image_path=/kaggle/working/real_input "
        "output_folder=/kaggle/working/real_output "
        "image_file_ending=nii.gz device=cuda:0 batch_size=1 overlap=0.25"
    )
    print("Exit code:", exit_code)

print("Contents of real_output:", os.listdir("/kaggle/working/real_output"))

In [ ]:
!nvidia-smi

## orig/pred load + sanity check

In [ ]:
orig = nib.load("/kaggle/working/real_input/kidney_K292_163um.nii.gz").get_fdata()
pred = nib.load("/kaggle/working/real_output/kidney_K292_163um_pred.nii.gz").get_fdata()

print("orig shape:", orig.shape)
print("pred shape:", pred.shape)
print("pred value range:", pred.min(), "-", pred.max())
print("predicted vessel voxels (>0.5):", np.sum(pred > 0.5))

## Cell 4 — SETUP CELL (collapse detection + adapter + loss + dataset class)

In [ ]:
import subprocess
result = subprocess.run(
    ["pip", "install", "--force-reinstall", "numpy==1.26.4"],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(result.stderr[-2000:])

In [ ]:
!pip install -q --force-reinstall --no-cache-dir "numpy==1.26.4" "scipy==1.13.1" "scikit-image==0.24.0" "monai==1.3.2"

In [ ]:
import sys
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from skimage.measure import regionprops, label
from skimage.filters import sato
from scipy import ndimage
from scipy.ndimage import binary_dilation

sys.path.insert(0, "/kaggle/working/vesselFM")
import hydra
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_cfg = OmegaConf.load("/kaggle/working/vesselFM/vesselfm/seg/configs/model/dyn_unet_base.yaml")
ckpt = torch.load(
    hf_hub_download(repo_id='bwittmann/vesselFM', filename='vesselFM_base.pt'),
    map_location=device, weights_only=True
)
model = hydra.utils.instantiate(model_cfg)
model.load_state_dict(ckpt)
model.to(device)
model.eval()
print("vesselFM model loaded.")


def compute_collapse_score(img_slice, percentile_threshold=85, min_area=5000):
    threshold = np.percentile(img_slice, percentile_threshold)
    mask = img_slice > threshold
    labeled_mask = label(mask)
    regions = regionprops(labeled_mask)

    dist_transform = ndimage.distance_transform_edt(mask)
    vesselness_map = sato(img_slice, sigmas=range(1, 6), black_ridges=False)

    scores = []
    for r in regions:
        if r.area <= min_area:
            continue
        region_mask = labeled_mask == r.label
        eccentricity = r.eccentricity
        local_thickness = dist_transform[region_mask]
        thickness_ratio = (local_thickness.std() / (local_thickness.mean() + 1e-8)
                            if local_thickness.size > 0 else np.nan)
        major = r.major_axis_length
        minor = r.minor_axis_length if r.minor_axis_length > 0 else 1e-8
        hessian_ratio = major / minor
        vesselness_val = vesselness_map[region_mask].mean()

        norm_eccentricity = eccentricity
        norm_thickness = min(thickness_ratio / 1.0, 1.0) if not np.isnan(thickness_ratio) else np.nan
        norm_hessian = min(hessian_ratio / 10.0, 1.0)
        norm_vesselness = 1.0 - min(vesselness_val * 10, 1.0)

        collapse_score = np.nanmean([norm_eccentricity, norm_thickness, norm_hessian, norm_vesselness])
        scores.append({
            "label": r.label, "eccentricity": eccentricity,
            "thickness_ratio": thickness_ratio, "hessian_ratio": hessian_ratio,
            "vesselness": vesselness_val, "collapse_score": collapse_score,
        })
    return scores, labeled_mask


class PhysicsAwareAdapter(nn.Module):
    def __init__(self, in_channels, bottleneck_channels=16):
        super().__init__()
        self.down = nn.Conv3d(in_channels + 1, bottleneck_channels, kernel_size=1)
        self.act = nn.GELU()
        self.spatial = nn.Conv3d(bottleneck_channels, bottleneck_channels,
                                  kernel_size=3, padding=1, groups=bottleneck_channels)
        self.up = nn.Conv3d(bottleneck_channels, in_channels, kernel_size=1)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, feat, collapse_score_map):
        collapse_score_map = F.interpolate(
            collapse_score_map, size=feat.shape[2:], mode="trilinear", align_corners=False
        )
        x = torch.cat([feat, collapse_score_map], dim=1)
        x = self.act(self.down(x))
        x = self.act(self.spatial(x))
        delta = self.up(x)
        return feat + delta


class CollapseAwareLoss(nn.Module):
    def __init__(self, dice_weight=1.0, bce_weight=1.0, shape_weight=0.5, smooth=1e-5):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.shape_weight = shape_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss(reduction="none")

    def dice_loss(self, probs, target):
        dims = (2, 3, 4)
        intersection = (probs * target).sum(dim=dims)
        union = probs.sum(dim=dims) + target.sum(dim=dims)
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()

    def shape_penalty(self, probs, target, collapse_score_map):
        dz = torch.abs(probs[:, :, 1:, :, :] - probs[:, :, :-1, :, :])
        dy = torch.abs(probs[:, :, :, 1:, :] - probs[:, :, :, :-1, :])
        dx = torch.abs(probs[:, :, :, :, 1:] - probs[:, :, :, :, :-1])
        weight = 1.0 - collapse_score_map
        tv = ((dz * weight[:, :, 1:, :, :]).mean()
              + (dy * weight[:, :, :, 1:, :]).mean()
              + (dx * weight[:, :, :, :, 1:]).mean())
        return tv

    def forward(self, logits, target, collapse_score_map):
        probs = torch.sigmoid(logits)
        bce = self.bce(logits, target).mean()
        dice = self.dice_loss(probs, target)
        shape = self.shape_penalty(probs, target, collapse_score_map)
        total = self.bce_weight * bce + self.dice_weight * dice + self.shape_weight * shape
        return total, {"bce": bce.item(), "dice": dice.item(), "shape": shape.item()}


class TiffKidneyVesselDataset(Dataset):
    def __init__(self, kidney_dir, patch_size=(32, 128, 128),
                 percentile_threshold=85, min_area=2000):
        self.kidney_dir = Path(kidney_dir)
        self.image_dir = self.kidney_dir / "images"
        self.label_dir = self.kidney_dir / "labels"
        self.image_files = sorted(self.image_dir.glob("*.tif"))
        self.label_files = sorted(self.label_dir.glob("*.tif"))
        assert len(self.image_files) == len(self.label_files), "image/label count mismatch"
        self.patch_depth, self.patch_h, self.patch_w = patch_size
        self.percentile_threshold = percentile_threshold
        self.min_area = min_area
        self.n_slices = len(self.image_files)

    def __len__(self):
        return max(self.n_slices - self.patch_depth, 1)

    def _load_slice(self, idx):
        img = np.array(Image.open(self.image_files[idx])).astype(np.float32)
        lbl = np.array(Image.open(self.label_files[idx])).astype(np.float32)
        return img, (lbl > 0).astype(np.float32)

    def __getitem__(self, idx):
        z0, z1 = idx, idx + self.patch_depth
        images, labels, collapse_scores = [], [], []
        for z in range(z0, z1):
            img, lbl = self._load_slice(z)
            scores, labeled_mask = compute_collapse_score(
                img, percentile_threshold=self.percentile_threshold, min_area=self.min_area
            )
            score_slice = np.zeros_like(img, dtype=np.float32)
            for s in scores:
                if np.isnan(s["collapse_score"]):
                    continue
                score_slice[labeled_mask == s["label"]] = s["collapse_score"]
            images.append(img); labels.append(lbl); collapse_scores.append(score_slice)

        image_vol = np.stack(images, axis=0)
        label_vol = np.stack(labels, axis=0)
        collapse_vol = np.stack(collapse_scores, axis=0)
        image_vol = (image_vol - image_vol.mean()) / (image_vol.std() + 1e-8)

        H, W = image_vol.shape[1:]
        ph, pw = min(self.patch_h, H), min(self.patch_w, W)
        y0 = np.random.randint(0, H - ph + 1)
        x0 = np.random.randint(0, W - pw + 1)
        image_vol = image_vol[:, y0:y0+ph, x0:x0+pw]
        label_vol = label_vol[:, y0:y0+ph, x0:x0+pw]
        collapse_vol = collapse_vol[:, y0:y0+ph, x0:x0+pw]

        return {
            "image": torch.from_numpy(image_vol).unsqueeze(0).float(),
            "mask": torch.from_numpy(label_vol).unsqueeze(0).float(),
            "collapse_score": torch.from_numpy(collapse_vol).unsqueeze(0).float(),
        }


print("Setup complete: compute_collapse_score, PhysicsAwareAdapter, CollapseAwareLoss, TiffKidneyVesselDataset ready.")

## Step 5 -- load the real ground-truth dataset (Kaggle SenNet+HOA) and sanity-check it

In [ ]:
train_dataset = TiffKidneyVesselDataset(
    "/kaggle/input/competitions/blood-vessel-segmentation/train/kidney_1_dense",
    patch_size=(128, 128, 128),
    min_area=50,
)
print("Number of image files found:", len(train_dataset.image_files))

sample = train_dataset[1000]
print("image:", sample["image"].shape)
print("mask vessel voxels:", sample["mask"].sum().item())
print("collapse_score max:", sample["collapse_score"].max().item())

In [ ]:
adapter = PhysicsAwareAdapter(in_channels=1, bottleneck_channels=16).to(device)
criterion = CollapseAwareLoss(dice_weight=1.0, bce_weight=1.0, shape_weight=0.5)
optimizer = torch.optim.AdamW(adapter.parameters(), lr=1e-3, weight_decay=1e-5)

n_trainable = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
print(f"Adapter trainable parameters: {n_trainable:,}")

In [ ]:
def run_backbone(image_patch):
    """Frozen forward pass through vesselFM on a single patch, returns raw logits (1,1,D,H,W)."""
    with torch.no_grad():
        logits = model(image_patch)
        if isinstance(logits, (list, tuple)):
            logits = logits[0]
    return logits


def train_one_epoch(dataset, adapter, criterion, optimizer, n_steps=50):
    adapter.train()
    losses = []
    for step in range(n_steps):
        idx = np.random.randint(0, len(dataset))
        sample = dataset[idx]
        image = sample["image"].unsqueeze(0).to(device)
        target = sample["mask"].unsqueeze(0).to(device)
        collapse_map = sample["collapse_score"].unsqueeze(0).to(device)

        backbone_logits = run_backbone(image)
        adapted_logits = adapter(backbone_logits, collapse_map)

        loss, parts = criterion(adapted_logits, target, collapse_map)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return float(np.mean(losses))


# quick test: just 1 step first, to catch shape errors early
test_loss = train_one_epoch(train_dataset, adapter, criterion, optimizer, n_steps=1)
print("Single test step loss:", test_loss)

In [ ]:
n_epochs = 5
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

for epoch in range(n_epochs):
    avg_loss = train_one_epoch(train_dataset, adapter, criterion, optimizer, n_steps=50)
    print(f"Epoch {epoch+1}/{n_epochs} -- avg loss: {avg_loss:.4f}")
    torch.save(adapter.state_dict(), "/kaggle/working/checkpoints/adapter_latest.pt")

print("Training complete. Checkpoint saved.")

## Next (not yet implemented)
1. Real few-shot training loop -- hook the adapter into vesselFM's actual encoder/decoder (need to inspect `print(model)` first to find the real layer names).
2. Before/after Dice comparison on kidney_1_dense (held-out slices) and zero-shot on kidney_2.
3. Baseline comparison (nnU-Net), statistical significance (CI, p-value across donors).
4. Murray's Law + vascular graph biological validation.
